# 07 · Total cost of ownership and the negotiation playbook

**TCO per unit** = landed price (freight, then import duty outside Mercosur) + inventory carried over the lead time plus 15 days of safety stock, at 20% a year + quality cost (PPM × landed price plus USD 200 per defect).

**Playbook** per flagged supplier: its BATNA is the lowest-TCO other source for the same items; the zone of possible agreement runs from the supplier's floor (its cost without margin) to the buyer's walk-away (the price at which switching becomes cheaper). Code in `src/tco.py` and `src/playbook.py`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
from datetime import date

import pandas as pd

from src import playbook, tco
from src.db import DEFAULT_DB, connect, create_schema
from src.figures import tco_by_plant

con = connect(DEFAULT_DB)
create_schema(con)
since = date(2025, 11, 1)
pairs = tco.pair_tco(con, since)

## TCO above price, by plant country

In [ ]:
tco_by_plant(con);

In [ ]:
tco.supplier_tco(pairs).groupby("country").agg(
    suppliers=("supplier_id", "count"),
    spend_usd_m=("spend", lambda s: round(s.sum() / 1e6, 2)),
    median_tco_uplift=("tco_uplift", "median"),
).sort_values("median_tco_uplift").round(3)

## Cheapest price vs. cheapest TCO

For items bought from more than one supplier, how often does the cheapest price lose on TCO?

In [ ]:
multi = pairs.groupby("material_id").filter(lambda g: len(g) > 1)
best_price = multi.loc[multi.groupby("material_id")["price"].idxmin()].set_index("material_id")
best_tco = multi.loc[multi.groupby("material_id")["tco"].idxmin()].set_index("material_id")
differ = best_price["supplier_id"] != best_tco["supplier_id"]
print(f"{differ.sum()} of {len(differ)} multi-source items: cheapest price is not cheapest TCO")
pd.DataFrame(
    {
        "cheapest_price_plant": best_price.loc[differ, "country"],
        "cheapest_tco_plant": best_tco.loc[differ, "country"],
    }
).value_counts().head(10)

Moving every item's volume to its lowest-TCO source is only an upper bound: it would undo the dual sourcing the category strategy asks for on strategic items.

In [ ]:
current = (multi["qty"] * multi["tco"]).sum()
bound = (
    multi.groupby("material_id")
    .apply(lambda g: g["qty"].sum() * g["tco"].min(), include_groups=False)
    .sum()
)
print(
    f"Upper bound, all volume to the lowest-TCO source: USD {current - bound:,.0f} "
    f"({1 - bound / current:.1%} of TCO spend on multi-source items)"
)

## Negotiation zones for the flagged suppliers

In [ ]:
rows = []
for s in playbook.targets(con):
    z = playbook.zopa(con, s, since)
    rows.append(
        {
            "supplier_id": s,
            "spend_usd": round(z["current"]),
            "floor": z["floor"] / z["current"] - 1,
            "should_cost": z["target"] / z["current"] - 1,
            "walk_away": z["walk_away"] / z["current"] - 1,
            "aim": z["aim"] / z["current"] - 1,
            "opening": z["opening"] / z["current"] - 1,
            "zopa": z["has_zopa"],
        }
    )
pd.DataFrame(rows).round(3)

The full playbook, with concessions and asks for each supplier, is written to `reports/negotiation_playbook.md` by `python -m src.playbook`.

## Reading

- Parts from outside Mercosur cost **26–33% more than their price** once freight, duty, inventory and quality are counted; Brazilian and Paraguayan plants add 6–9%. That gap is why Paraguay dominates Brazil's harness imports.
- In **about half of the multi-source items** the cheapest price is not the cheapest TCO: price-only awards would be wrong half the time.
- **S003** (German group, plant in Morocco) has a real zone of agreement: aim at should-cost, 13% below today, with a 3-year agreement and the quarterly copper clause as the main concessions.
- For **S027** the alternatives already beat should-cost: aim at the walk-away, 16% below today.
- **S023 and S006**, both Japanese plants, have no zone: the alternatives' TCO is below their own floor. Move volume in stages and use should-cost only as the price for the transition.